# NLP: Fine-tuning DistilBERT на вопросах Polymarket

Предсказание outcome рынка только по тексту вопроса — некоррелированный сигнал для ensemble с табличным HTR.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
torch.set_num_threads(1)
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics import roc_auc_score, classification_report
from torch.utils.data import Dataset, DataLoader

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA = Path('../../data/processed')
MODELS = Path('../../data/models')
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')

## 1. Данные и tokenization

In [ ]:
# Загрузка данных
outcomes = pd.read_parquet(DATA / 'resolution_outcomes.parquet')
df = outcomes[outcomes['outcome'].isin([0, 1])].copy()
df = df.dropna(subset=['closedTime']).sort_values('closedTime')

print(f'Resolved markets: {len(df)}')
print(f'Outcome: 0={( df.outcome==0).sum()}, 1={(df.outcome==1).sum()}')
print(f'Sample: {df.question.iloc[0][:80]}')

In [ ]:
# Tokenizer — как BERT читает текст
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Пример tokenization
example = "Will Trump win the 2024 presidential election?"
tokens = tokenizer(example, return_tensors='pt')
print(f'Text: {example}')
print(f'Token IDs: {tokens["input_ids"][0].tolist()}')
print(f'Decoded: {tokenizer.decode(tokens["input_ids"][0])}')
print(f'Tokens: {tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])}')

In [ ]:
# Time-based split
n = len(df)
train_df = df.iloc[:int(n * 0.7)]
val_df = df.iloc[int(n * 0.7):int(n * 0.85)]
test_df = df.iloc[int(n * 0.85):]

# Subsample для скорости обучения (322K — много для fine-tuning)
MAX_TRAIN = 50_000
MAX_VAL = 10_000
MAX_TEST = 10_000

if len(train_df) > MAX_TRAIN:
    train_df = train_df.sample(MAX_TRAIN, random_state=SEED)
if len(val_df) > MAX_VAL:
    val_df = val_df.sample(MAX_VAL, random_state=SEED)
if len(test_df) > MAX_TEST:
    test_df = test_df.sample(MAX_TEST, random_state=SEED)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

In [ ]:
class QuestionDataset(Dataset):
    """Dataset: текст вопроса → outcome (0/1)"""
    def __init__(self, questions, labels, tokenizer, max_len=64):
        self.questions = questions
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.questions)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.questions[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.float)
        }

train_ds = QuestionDataset(train_df['question'].tolist(), train_df['outcome'].values, tokenizer)
val_ds = QuestionDataset(val_df['question'].tolist(), val_df['outcome'].values, tokenizer)
test_ds = QuestionDataset(test_df['question'].tolist(), test_df['outcome'].values, tokenizer)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
test_loader = DataLoader(test_ds, batch_size=64)

print(f'Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

## 2. Fine-tuning DistilBERT

Заморозка 4 из 6 transformer layers, дообучение верхних слоёв + classifier head.

In [ ]:
class MarketBERT(nn.Module):
    """DistilBERT + classification head для предсказания outcome."""
    def __init__(self, freeze_bert_layers=4):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        
        # Заморозить нижние слои (они уже знают язык)
        # DistilBERT имеет 6 transformer layers
        for i, layer in enumerate(self.bert.transformer.layer):
            if i < freeze_bert_layers:
                for param in layer.parameters():
                    param.requires_grad = False
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
    
    def forward(self, input_ids, attention_mask):
        # [CLS] token embedding — summary of the whole sentence
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = output.last_hidden_state[:, 0, :]  # (batch, 768)
        return self.classifier(cls_embedding).squeeze(-1)
    
    def get_embedding(self, input_ids, attention_mask):
        """Extract 768-dim embedding (для ensemble)"""
        with torch.no_grad():
            output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            return output.last_hidden_state[:, 0, :]

model = MarketBERT(freeze_bert_layers=4).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Total params: {total:,} | Trainable: {trainable:,} ({trainable/total*100:.1f}%)')

In [ ]:
optimizer = torch.optim.AdamW([
    {'params': model.bert.parameters(), 'lr': 2e-5},      # BERT — маленький LR
    {'params': model.classifier.parameters(), 'lr': 1e-3}, # Head — побольше
], weight_decay=0.01)

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=3)

@torch.no_grad()
def evaluate_bert(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        logits = model(ids, mask)
        all_preds.append(torch.sigmoid(logits).cpu())
        all_labels.append(batch['label'])
    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    auc = roc_auc_score(labels, preds)
    return auc, preds, labels

In [ ]:
# Training loop — 3 epochs (для fine-tuning больше не нужно)
EPOCHS = 3
best_auc = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for i, batch in enumerate(train_loader):
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        total_loss += loss.item()
        
        if (i + 1) % 200 == 0:
            print(f'  batch {i+1}/{len(train_loader)} loss={loss.item():.4f}')
    
    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    val_auc, _, _ = evaluate_bert(model, val_loader)
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), MODELS / 'market_bert_v1.pt')
    
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | '
          f'Val AUC: {val_auc:.4f} {"★" if val_auc >= best_auc else ""}')

print(f'\nBest Val AUC: {best_auc:.4f}')

In [ ]:
# Test evaluation
model.load_state_dict(torch.load(MODELS / 'market_bert_v1.pt', weights_only=True))
test_auc, test_preds, test_labels = evaluate_bert(model, test_loader)

print(f'Test AUC: {test_auc:.4f}')
print(f'\nClassification report (threshold=0.5):')
print(classification_report(test_labels, (test_preds > 0.5).astype(int),
                          target_names=['NO (0)', 'YES (1)']))

## Результаты

| Подход | Val AUC | Test AUC | Input |
|--------|---------|----------|-------|
| MLP (5 табличных фич) | 0.6284 | 0.6437 | volume, liquidity, spread |
| LightGBM (5 табл фич) | 0.6159 | 0.6259 | те же |
| **DistilBERT (текст)** | **0.6315** | **0.6499** | **только текст вопроса** |
| HTR (14 mid-life фич) | — | 0.9591 | rich табличные фичи |

NLP-модель даёт независимый сигнал — текст не коррелирует с табличными фичами.

In [ ]:
# Predictions by category
import re

def categorize(q):
    q = q.lower()
    if any(x in q for x in ['bitcoin', 'ethereum', 'solana', 'crypto', 'btc', 'eth']): return 'crypto'
    if any(x in q for x in ['trump', 'biden', 'election', 'president']): return 'politics'
    if any(x in q for x in ['o/u', 'over', 'under', 'points', 'assists']): return 'sports_ou'
    if any(x in q for x in ['vs.', 'vs ']): return 'sports_match'
    if 'up or down' in q: return 'price_direction'
    return 'other'

test_df_eval = test_df.copy()
test_df_eval['pred'] = test_preds
test_df_eval['category'] = test_df_eval['question'].apply(categorize)

print(f'{"Category":<20} {"N":>6} {"AUC":>8} {"Avg Pred":>10} {"True Rate":>10}')
print('-' * 60)
for cat, grp in test_df_eval.groupby('category'):
    if len(grp) > 50:
        try:
            auc = roc_auc_score(grp['outcome'], grp['pred'])
        except:
            auc = 0.5
        print(f'{cat:<20} {len(grp):>6} {auc:>8.4f} {grp.pred.mean():>10.3f} {grp.outcome.mean():>10.3f}')

In [ ]:
# Самые уверенные предсказания
print('=== TOP-10 уверенных YES (pred > 0.8) ===')
top_yes = test_df_eval.nlargest(10, 'pred')
for _, r in top_yes.iterrows():
    mark = '✓' if r.outcome == 1 else '✗'
    print(f'  {mark} p={r.pred:.3f} [{r.outcome}] {r.question[:70]}')

print(f'\n=== TOP-10 уверенных NO (pred < 0.2) ===')
top_no = test_df_eval.nsmallest(10, 'pred')
for _, r in top_no.iterrows():
    mark = '✓' if r.outcome == 0 else '✗'
    print(f'  {mark} p={r.pred:.3f} [{r.outcome}] {r.question[:70]}')

## 3. Извлечение NLP embeddings для ensemble

In [ ]:
# Extract embeddings for all test data
model.eval()
embeddings = []
with torch.no_grad():
    for batch in test_loader:
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        emb = model.get_embedding(ids, mask)
        embeddings.append(emb.cpu())

embeddings = torch.cat(embeddings).numpy()
print(f'Embeddings shape: {embeddings.shape}')  # (N, 768)

# PCA для визуализации
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1], 
                     c=test_labels, cmap='RdYlGn', alpha=0.3, s=5)
plt.colorbar(scatter, label='Outcome')
ax.set_title('BERT Embeddings (PCA) — colored by outcome')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.tight_layout(); plt.show()

print(f'\nEmbeddings можно использовать как фичи в HTR ensemble.')
print(f'Сохраняем для будущего использования.')
np.save(MODELS / 'bert_embeddings_test.npy', embeddings)